# Lung Nematic

Graphical front-end over the **same engine** as the command line
(`lung_nematic.batch.analyze_folder` → `analyze_image`). Whatever you run here
produces the identical outputs a CLI run would: per-image overlays, defect and
nucleus tables, null-model and colocalization results, and a metrics summary.

Run the cells top to bottom. The code is hidden — you only see titles and
controls.

In [ ]:
#@title 🔧 Setup — run once { display-mode: "form" }
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "git+https://github.com/Danpc11/lung-nematic.git"], check=True)
import lung_nematic
try:
    from importlib.metadata import version
    _v = version("lung-nematic")
except Exception:
    _v = getattr(lung_nematic, "__version__", "unknown")
print(f"Ready. lung_nematic {_v}")

In [ ]:
#@title 📁 Load images — upload or from Drive { display-mode: "form" }
image_source = "Upload from computer" #@param ["Upload from computer", "Google Drive folder"]
drive_folder = "MyDrive/lung_histology" #@param {type:"string"}
import os, shutil, glob
IMAGES_DIR = "/content/images"
EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")
if os.path.isdir(IMAGES_DIR): shutil.rmtree(IMAGES_DIR)
os.makedirs(IMAGES_DIR, exist_ok=True)
if image_source == "Upload from computer":
    from google.colab import files
    for name in files.upload():
        shutil.move(name, os.path.join(IMAGES_DIR, os.path.basename(name)))
else:
    from google.colab import drive
    if not os.path.ismount("/content/drive"): drive.mount("/content/drive")
    src = drive_folder if os.path.isabs(drive_folder) else os.path.join("/content/drive", drive_folder)
    if not os.path.isdir(src): raise FileNotFoundError(f"Drive folder not found: {src}")
    found = [p for p in sorted(glob.glob(os.path.join(src, "*"))) if p.lower().endswith(EXTS)]
    if not found: raise FileNotFoundError(f"No {EXTS} images in {src}")
    for p in found: shutil.copy(p, os.path.join(IMAGES_DIR, os.path.basename(p)))
names = sorted(os.listdir(IMAGES_DIR))
print(f"{len(names)} image(s) ready:")
for n in names: print("  ", n)

In [ ]:
#@title ⚙️ Parameters { display-mode: "form", run: "auto" }
#@markdown **Field & analyses**
field_type = "collagen" #@param ["nuclear", "collagen", "fused"]
run_null = True #@param {type:"boolean"}
run_colocalization = True #@param {type:"boolean"}
#@markdown **Director field**
sigmas_px = "40, 55, 70, 85" #@param {type:"string"}
defect_grid_step_px = 24 #@param {type:"slider", min:8, max:64, step:4}
min_edge_distance_px = 30 #@param {type:"slider", min:0, max:120, step:5}
density_quantile = 0.45 #@param {type:"slider", min:0, max:0.9, step:0.05}
min_scales_for_persistence = 2 #@param {type:"slider", min:1, max:4, step:1}
defect_cluster_radius_px = 70 #@param {type:"slider", min:20, max:150, step:5}
detect_integer_defects = False #@param {type:"boolean"}
integer_defect_loop_radius_px = 30 #@param {type:"slider", min:8, max:80, step:2}
integer_defect_loop_points = 8 #@param {type:"slider", min:6, max:24, step:1}
save_defect_maps = False #@param {type:"boolean"}
defect_map_window_px = 220 #@param {type:"slider", min:80, max:400, step:20}
mask_normalized_smoothing = False #@param {type:"boolean"}
collagen_inner_scale_px = 1.5 #@param {type:"number"}
#@markdown **Null model**
n_permutations = 199 #@param {type:"slider", min:19, max:999, step:10}
null_downsample = 2 #@param {type:"slider", min:1, max:6, step:1}
null_mode = "shuffle" #@param ["shuffle", "uniform"]
#@markdown **Colocalization**
n_bootstrap = 2000 #@param {type:"slider", min:200, max:5000, step:100}
#@markdown **Scale & reproducibility**
microns_per_pixel = 0.0 #@param {type:"number"}
random_seed = 42 #@param {type:"integer"}

from lung_nematic.config import AnalysisConfig
try:
    _sigmas = tuple(float(s) for s in sigmas_px.split(",") if s.strip())
    config = AnalysisConfig(
        sigmas_px=_sigmas,
        defect_grid_step_px=int(defect_grid_step_px),
        min_edge_distance_px=int(min_edge_distance_px),
        density_quantile=float(density_quantile),
        min_scales_for_persistence=min(int(min_scales_for_persistence), len(_sigmas)),
        defect_cluster_radius_px=float(defect_cluster_radius_px),
        detect_integer_defects=bool(detect_integer_defects),
        integer_defect_loop_radius_px=int(integer_defect_loop_radius_px),
        integer_defect_loop_points=int(integer_defect_loop_points),
        save_defect_maps=bool(save_defect_maps),
        defect_map_window_px=int(defect_map_window_px),
        mask_normalized_smoothing=bool(mask_normalized_smoothing),
        collagen_inner_scale_px=float(collagen_inner_scale_px),
        field_type=field_type, run_null=bool(run_null),
        run_colocalization=bool(run_colocalization),
        n_permutations=int(n_permutations), null_downsample=int(null_downsample),
        null_mode=null_mode, n_bootstrap=int(n_bootstrap),
        default_microns_per_pixel=(microns_per_pixel or None),
        random_seed=int(random_seed),
    )
    config.validate()
    print(f"Config ready — field={config.field_type}, null={config.run_null}, "
          f"colocalization={config.run_colocalization}")
except Exception as error:
    print("Adjust parameters:", error)

In [ ]:
#@title ▶️ Run analysis { display-mode: "form" }
import warnings; warnings.filterwarnings("ignore")
from lung_nematic.batch import analyze_folder
summary, errors = analyze_folder("/content/images", "/content/results", config=config)
print(f"Processed {len(summary)} image(s), {len(errors)} failed.")
if not errors.empty:
    display(errors)
summary

In [ ]:
#@title 📊 Show results { display-mode: "form" }
import glob, os
import pandas as pd
from IPython.display import Image as IPyImage, display
for overlay in sorted(glob.glob("/content/results/*/*_overlay.png")):
    folder = os.path.dirname(overlay)
    print("=" * 60); print(os.path.basename(folder))
    display(IPyImage(overlay, width=880))
    for hist in sorted(glob.glob(os.path.join(folder, "*_null_hist.png"))):
        display(IPyImage(hist, width=520))
    for dmap in sorted(glob.glob(os.path.join(folder, "defect_maps", "*.png")))[:12]:
        display(IPyImage(dmap, width=420))
sp = "/content/results/summary_metrics.csv"
if os.path.exists(sp):
    print("Summary:"); display(pd.read_csv(sp))

In [ ]:
#@title ⬇️ Download results (zip) { display-mode: "form" }
import shutil
from google.colab import files
archive = shutil.make_archive("/content/lung_nematic_results", "zip", "/content/results")
files.download(archive)